In [39]:
# 도서정보객체
class Book:
    def __init__(self, rank, title, author, price):
        self.rank = rank
        self.title = title
        self.author = author
        self.price = price

    def __str__(self):
        return f"{self.rank}, {self.title}, {self.author}, {self.price}"
    
    def to_dict(self):
        return {'rank':self.rank, 
                'title':self.title, 
                'author':self.author, 
                'price':self.price}
    
    def to_list(self):
        return [self.rank, 
                self.title, 
                self.author, 
                self.price]

In [40]:
# iteminfo
# yesBestList > li:nth-child(1) > div > div.item_info
import requests
from bs4 import BeautifulSoup
res = 'https://www.yes24.com/Product/Category/BestSeller?categoryNumber=001'
response = requests.get(res)
soup = BeautifulSoup(response.content, 'html.parser')
best_list_el = soup.select('#yesBestList div.item_info')
len(best_list_el)

24

In [41]:
book_list =[]
for i, item in enumerate(best_list_el):
    title = item.select_one('div.info_name > a').text
    author = item.select_one('.info_auth > a').text
    price = item.select_one('.info_price .yes_b').text
    book_list.append(Book(i+1, title, author, price))
for book in book_list:
    print(book)


1, 혼모노, 성해나, 16,200
2, 류수영의 평생 레시피, 류수영, 22,500
3, 가공범, 히가시노 게이고, 19,800
4, 돌비공포라디오 더 레드, 돌비, 17,820
5, 2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상, 최태성, 14,850
6, 박곰희 연금 부자 수업, 박곰희, 18,900
7, 2025 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심화(1,2,3급), 최태성, 17,550
8, 단 한 줄만 내 마음에 새긴다고 해도, 나민애, 21,420
9, 2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하, 최태성, 14,400
10, ETS 토익 정기시험 기출문제집 1000 Vol. 4 RC, ETS, 17,820
11, ETS 토익 정기시험 기출문제집 1000 Vol. 4 LC, ETS, 17,820
12, 료의 생각 없는 생각, 료, 18,000
13, 모순, 양귀자, 11,700
14, 안녕이라 그랬어, 김애란, 15,120
15, 자몽살구클럽, 한로로, 10,800
16, 인생을 바꾸는 최고의 ETF, 잼투리, 22,050
17, 청춘의 독서, 유시민, 17,010
18, 소년이 온다, 한강, 13,500
19, 첫 여름, 완주, 김금희, 15,300
20, 경험의 멸종, 크리스틴 로젠, 17,820
21, 어른의 행복은 조용하다, 태수, 16,020
22, 견우와 선녀 대본집 세트, 양지훈, 41,400
23, 어른의 품격을 채우는 100일 필사 노트, 김종원, 18,000
24, 야구선수 김원중, 김원중, 18,000


# sqlite

In [42]:
import sqlite3
conn = sqlite3.connect('my_database.db')
cursor =conn.cursor()
cursor.execute("DROP TABLE IF EXISTS books")
query = '''
CREATE TABLE IF NOT EXISTS books (
    RANK INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author TEXT NOT NULL,
    price INTEGER
)
'''
cursor.execute(query)
conn.commit()


In [43]:
ins_query = '''
INSERT INTO books (RANK, title, author, price)
VALUES (?, ?, ?, ?)
'''


In [44]:
for book in book_list:
    cursor.execute(ins_query,book.to_list())

conn.commit()
conn.close()

# sqlite에서 mysql workbench로 데이터 이전

In [45]:
# SQLite 연결
sqlite_conn = sqlite3.connect('my_database.db')
sqlite_cursor = sqlite_conn.cursor()

In [46]:
# 쉼표 제거 후 price 정수로 변환해 데이터 조회
sqlite_cursor.execute("""
    SELECT RANK, title, author, CAST(REPLACE(price, ',', '') AS INTEGER)
    FROM books
""")
rows = sqlite_cursor.fetchall()
rows

[(1, '혼모노', '성해나', 16200),
 (2, '류수영의 평생 레시피', '류수영', 22500),
 (3, '가공범', '히가시노 게이고', 19800),
 (4, '돌비공포라디오 더 레드', '돌비', 17820),
 (5, '2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상', '최태성', 14850),
 (6, '박곰희 연금 부자 수업', '박곰희', 18900),
 (7, '2025 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심화(1,2,3급)', '최태성', 17550),
 (8, '단 한 줄만 내 마음에 새긴다고 해도', '나민애', 21420),
 (9, '2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하', '최태성', 14400),
 (10, 'ETS 토익 정기시험 기출문제집 1000 Vol. 4 RC', 'ETS', 17820),
 (11, 'ETS 토익 정기시험 기출문제집 1000 Vol. 4 LC', 'ETS', 17820),
 (12, '료의 생각 없는 생각', '료', 18000),
 (13, '모순', '양귀자', 11700),
 (14, '안녕이라 그랬어', '김애란', 15120),
 (15, '자몽살구클럽', '한로로', 10800),
 (16, '인생을 바꾸는 최고의 ETF', '잼투리', 22050),
 (17, '청춘의 독서', '유시민', 17010),
 (18, '소년이 온다', '한강', 13500),
 (19, '첫 여름, 완주', '김금희', 15300),
 (20, '경험의 멸종', '크리스틴 로젠', 17820),
 (21, '어른의 행복은 조용하다', '태수', 16020),
 (22, '견우와 선녀 대본집 세트', '양지훈', 41400),
 (23, '어른의 품격을 채우는 100일 필사 노트', '김종원', 18000),
 (24, '야구선수 김원중', '김원중', 18000)]

In [47]:
import mysql.connector

In [48]:
# MySQL 연결 (Workbench가 연결된 서버 정보 입력)
mysql_conn = mysql.connector.connect(
    host='localhost',         
    user='root',             
    password='1234', 
    database='bookstore'      
)
mysql_cursor = mysql_conn.cursor()

In [49]:
# MySQL에 데이터 삽입
mysql_cursor.execute("DELETE FROM books")
mysql_conn.commit()

insert_query = "INSERT INTO books (book_rank, title, author, price) VALUES (%s, %s, %s, %s)"
for row in rows:
    mysql_cursor.execute(insert_query, row)

In [50]:
# 커밋 및 연결 종료
mysql_conn.commit()
sqlite_conn.close()
mysql_conn.close()

# 여러페이지 가져오기

In [51]:
# base_url = 'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber='

In [52]:
import time

# SQLite DB와 테이블 생성 (없으면 생성)
conn = sqlite3.connect('my_database.db')
cursor = conn.cursor()

base_url = 'https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber='

for page in range(1, 6):
    url = base_url + str(page)
    print(f'크롤링 중: {url}')
    
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    best_list_el = soup.select('#yesBestList div.item_info')
    
    for i, item in enumerate(best_list_el):
        title = item.select_one('div.info_name > a').text.strip()
        author = item.select_one('div.info_pubGrp a').text.strip()
        price = item.select_one('.info_price > .txt_num').text.strip()
        
        rank = (page - 1) * 24 + i + 1
        
        cursor.execute('''
            INSERT OR REPLACE INTO books (rank, title, author, price)
            VALUES (?, ?, ?, ?)
        ''', (rank, title, author, price))
    
    conn.commit()
    time.sleep(1)  # 서버 과부하 방지

conn.close()
print("크롤링 및 DB 저장 완료")


크롤링 중: https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber=1
크롤링 중: https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber=2
크롤링 중: https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber=3
크롤링 중: https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber=4
크롤링 중: https://www.yes24.com/product/category/bestseller?categoryNumber=001&pageSize=24&pageNumber=5
크롤링 및 DB 저장 완료


In [53]:
import sqlite3

conn = sqlite3.connect('my_database.db')
sqlite_cursor = conn.cursor()

sqlite_cursor.execute("""
    SELECT RANK, title, author, CAST(REPLACE(price, ',', '') AS INTEGER) AS price_clean
    FROM books
""")

rows = sqlite_cursor.fetchall()
rows


[(1, '혼모노', '성해나', 16200),
 (2, '류수영의 평생 레시피', '류수영', 22500),
 (3, '가공범', '히가시노 게이고', 19800),
 (4, '돌비공포라디오 더 레드', '돌비', 17820),
 (5, '2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 상', '최태성', 14850),
 (6, '박곰희 연금 부자 수업', '박곰희', 18900),
 (7, '2025 큰별쌤 최태성의 별별한국사 기출 500제 한국사능력검정시험 심화(1,2,3급)', '최태성', 17550),
 (8, '단 한 줄만 내 마음에 새긴다고 해도', '나민애', 21420),
 (9, '2025 큰별쌤 최태성의 별별한국사 한국사능력검정시험 심화(1,2,3급) 하', '최태성', 14400),
 (10, 'ETS 토익 정기시험 기출문제집 1000 Vol. 4 RC', 'ETS', 17820),
 (11, 'ETS 토익 정기시험 기출문제집 1000 Vol. 4 LC', 'ETS', 17820),
 (12, '료의 생각 없는 생각', '료', 18000),
 (13, '모순', '양귀자', 11700),
 (14, '안녕이라 그랬어', '김애란', 15120),
 (15, '자몽살구클럽', '한로로', 10800),
 (16, '인생을 바꾸는 최고의 ETF', '잼투리', 22050),
 (17, '청춘의 독서', '유시민', 17010),
 (18, '소년이 온다', '한강', 13500),
 (19, '첫 여름, 완주', '김금희', 15300),
 (20, '경험의 멸종', '크리스틴 로젠', 17820),
 (21, '어른의 행복은 조용하다', '태수', 16020),
 (22, '견우와 선녀 대본집 세트', '양지훈', 41400),
 (23, '어른의 품격을 채우는 100일 필사 노트', '김종원', 18000),
 (24, '야구선수 김원중', '김원중', 18000),
 (25, '다크 심리학', '다크 사이드 프

In [54]:
# MySQL 연결 (Workbench가 연결된 서버 정보 입력)
mysql_conn = mysql.connector.connect(
    host='localhost',         
    user='root',             
    password='1234', 
    database='bookstore'      
)
mysql_cursor = mysql_conn.cursor()

In [55]:
insert_query = "REPLACE INTO books (book_rank, title, author, price) VALUES (%s, %s, %s, %s)"
for row in rows:
    mysql_cursor.execute(insert_query, row)


In [56]:
# 커밋 및 연결 종료
mysql_conn.commit()
sqlite_conn.close()
mysql_conn.close()
